# Spine-Re：重新训练与可复核指标

使用 Colab T4 GPU。本笔记本基于仓库现有 PNG 进行新实验，**不是论文的精确复现**。
原始 NIfTI 和患者映射缺失；未知标签颜色占约 1.6%–1.7%。默认忽略这些像素，无法恢复真实标注。
实际 J-Unet 参数 25,659,999，与论文 22,331,979 不同。

每项结果必须注明：训练轮数、标签策略、全局/逐图汇总、是否包含背景。
3 轮是短程实验，100 轮才对应论文表 2 的训练预算；不能仅凭跑通宣称复现。


In [ ]:
from pathlib import Path
import sys, subprocess, json, datetime
import torch
assert torch.cuda.is_available(), '请选择：运行时 → 更改运行时类型 → T4 GPU'
print('Python',sys.version,'PyTorch',torch.__version__,'GPU',torch.cuda.get_device_name(0))
REPO=Path('/content/Spine-Re-pinned')
SOURCE_COMMIT='b678a3130c7f2472208f33ce9c129ecc297061b0'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/B-jack7/Spine-Re.git',str(REPO)],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==SOURCE_COMMIT


## 安装修复代码并测试
以下代码直接包含本次修复版本，不依赖远端分支是否合并。原始模型和数据来自固定提交。

In [ ]:
payload=json.loads('{"audit.py": "\\"\\"\\"Audit supplied PNGs and recompute archived prediction metrics, without training.\\"\\"\\"\\nimport argparse\\nfrom collections import defaultdict\\nimport csv\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport numpy as np\\nfrom PIL import Image\\nfrom metrics import decode_mask, confusion_matrix, metrics\\n\\n\\ndef audit(repo, output):\\n    base = next(repo.glob(\'*/data\'))\\n    output.mkdir(parents=True, exist_ok=True)\\n    manifest, hashes, summaries = [], defaultdict(list), {}\\n    historical = {policy: np.zeros((3, 3), dtype=np.int64) for policy in (\'ignore\', \'legacy-background\')}\\n    historical_per_image = {policy: [] for policy in historical}\\n    predictions = next(repo.glob(\'*/J-Unet/result_pics\'))\\n    prediction_count = 0\\n    for split in (\'train\', \'val\', \'test\'):\\n        images = {p.name: p for p in (base/split).glob(\'*.png\')}\\n        labels = {p.name: p for p in (base/(split+\'_labels\')).glob(\'*.png\')}\\n        if images.keys() != labels.keys():\\n            raise ValueError(f\'{split}: image/label filenames do not match\')\\n        unknown = total = 0\\n        class_pixels = np.zeros(3, dtype=np.int64)\\n        sizes = set()\\n        for name in sorted(images, key=lambda n: int(Path(n).stem)):\\n            with Image.open(images[name]) as f:\\n                image_array = np.array(f.convert(\'RGB\'))\\n                size = f.size\\n            with Image.open(labels[name]) as f:\\n                raw_mask = np.array(f.convert(\'RGB\'))\\n            if raw_mask.shape != image_array.shape:\\n                raise ValueError(f\'{split}/{name}: shape mismatch\')\\n            target = decode_mask(raw_mask)\\n            digest = hashlib.sha256(str(image_array.shape).encode()+image_array.tobytes()).hexdigest()\\n            hashes[digest].append(f\'{split}/{name}\')\\n            sizes.add(size)\\n            bad = int((target < 0).sum())\\n            unknown += bad\\n            total += target.size\\n            class_pixels += np.bincount(target[target >= 0], minlength=3)\\n            manifest.append(dict(split=split, filename=name, image_sha256=digest,\\n                                 mask_sha256=hashlib.sha256(raw_mask.tobytes()).hexdigest(),\\n                                 unknown_pixels=bad, pixels=target.size, patient_id=\'unknown\'))\\n            if split == \'test\' and (predictions/name).exists():\\n                with Image.open(predictions/name) as f:\\n                    pred = decode_mask(f)\\n                if np.any(pred < 0):\\n                    raise ValueError(\'Archived prediction has unknown colors\')\\n                for policy in historical:\\n                    cm = confusion_matrix(pred, decode_mask(raw_mask, policy))\\n                    historical[policy] += cm\\n                    historical_per_image[policy].append(metrics(cm))\\n                prediction_count += 1\\n        summaries[split] = dict(images=len(images), sizes=sorted(sizes),\\n                                class_pixels=class_pixels.tolist(), unknown_pixels=unknown,\\n                                total_pixels=total, unknown_fraction=unknown/total)\\n    duplicates = [v for v in hashes.values() if len(v)>1]\\n    cross = [v for v in duplicates if len({p.split(\'/\')[0] for p in v})>1]\\n    old_scores = {}\\n    for policy, cm in historical.items():\\n        old_scores[policy] = metrics(cm)\\n        old_scores[policy][\'image_macro_averages\'] = {\\n            key: float(np.mean([v[key] for v in historical_per_image[policy] if v[key] is not None]))\\n            for key in (\'pixel_accuracy\',\'dice_macro_all\',\'miou_all\',\'dice_macro_foreground\',\\n                        \'miou_foreground\',\'legacy_mean_recall_background_vertebra\')}\\n    result = dict(source_commit=\'b678a3130c7f2472208f33ce9c129ecc297061b0\',\\n                  split_summary=summaries, exact_duplicate_groups=duplicates,\\n                  cross_split_duplicate_groups=cross,\\n                  patient_separation=\'unverifiable: patient IDs and original NIfTI absent\',\\n                  authenticity=\'not independently verified; repository PNGs only\',\\n                  archived_predictions=dict(count=prediction_count, origin=\'repository result_pics; weights absent; not a new model run\', scores=old_scores))\\n    (output/\'data_audit.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\\n    with (output/\'manifest.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\\n        w=csv.DictWriter(f,fieldnames=list(manifest[0])); w.writeheader(); w.writerows(manifest)\\n    print(json.dumps(result,indent=2))\\n    return result\\n\\n\\nif __name__ == \'__main__\':\\n    p=argparse.ArgumentParser(); p.add_argument(\'--repo\',type=Path,default=Path(__file__).resolve().parents[1]); p.add_argument(\'--output\',type=Path,required=True)\\n    a=p.parse_args(); audit(a.repo,a.output)\\n", "metrics.py": "\\"\\"\\"Explicit three-class metrics. Rows=truth, columns=prediction; -100=ignore.\\"\\"\\"\\nimport numpy as np\\n\\nCLASS_NAMES = [\'background\', \'vertebra\', \'disc\']\\n\\n\\ndef decode_mask(rgb, policy=\'ignore\'):\\n    rgb = np.asarray(rgb.convert(\'RGB\') if hasattr(rgb, \'convert\') else rgb)\\n    out = np.full(rgb.shape[:2], -100, dtype=np.int64)\\n    for cls, value in enumerate((0, 100, 255)):\\n        out[np.all(rgb == value, axis=-1)] = cls\\n    if policy == \'legacy-background\':\\n        out[out < 0] = 0\\n    elif policy != \'ignore\':\\n        raise ValueError(policy)\\n    return out\\n\\n\\ndef confusion_matrix(pred, target):\\n    pred, target = np.asarray(pred), np.asarray(target)\\n    if pred.shape != target.shape:\\n        raise ValueError(\'Prediction and target shapes differ\')\\n    keep = target != -100\\n    if not np.all((target[keep] >= 0) & (target[keep] < 3)):\\n        raise ValueError(\'Invalid target class\')\\n    if not np.all((pred[keep] >= 0) & (pred[keep] < 3)):\\n        raise ValueError(\'Invalid prediction class\')\\n    return np.bincount(3 * target[keep].astype(int) + pred[keep].astype(int), minlength=9).reshape(3, 3)\\n\\n\\ndef metrics(cm):\\n    cm = np.asarray(cm, dtype=np.float64)\\n    tp, truth, pred = np.diag(cm), cm.sum(1), cm.sum(0)\\n    def divide(a, b):\\n        return np.divide(a, b, out=np.full_like(a, np.nan), where=b != 0)\\n    dice = divide(2 * tp, truth + pred)\\n    iou = divide(tp, truth + pred - tp)\\n    recall = tp / (truth + 1e-10)\\n    def mean(x):\\n        return float(np.nanmean(x)) if np.any(np.isfinite(x)) else None\\n    def safe(x):\\n        return [float(v) if np.isfinite(v) else None for v in x]\\n    return dict(pixel_accuracy=float(tp.sum()/cm.sum()) if cm.sum() else None,\\n                dice_per_class=safe(dice), iou_per_class=safe(iou),\\n                dice_macro_all=mean(dice), dice_macro_foreground=mean(dice[1:]),\\n                miou_all=mean(iou), miou_foreground=mean(iou[1:]),\\n                # Historical trainer called this \\"accuracy\\", and excluded disc.\\n                legacy_mean_recall_background_vertebra=mean(recall[:2]),\\n                valid_pixels=int(cm.sum()), confusion=cm.astype(int).tolist())\\n", "test_core.py": "\\"\\"\\"Hand-checkable tests for labels, metric definitions and excluded pixels.\\"\\"\\"\\nimport unittest\\nimport numpy as np\\nfrom metrics import decode_mask, confusion_matrix, metrics\\n\\n\\nclass MetricTests(unittest.TestCase):\\n    def test_unknown_labels_are_not_silent_background(self):\\n        rgb=np.array([[[0,0,0],[100,100,100],[255,255,255],[99,99,99]]],dtype=np.uint8)\\n        np.testing.assert_array_equal(decode_mask(rgb),[[0,1,2,-100]])\\n        np.testing.assert_array_equal(decode_mask(rgb,\'legacy-background\'),[[0,1,2,0]])\\n\\n    def test_known_confusion_and_foreground(self):\\n        # Four background correct, one vertebra missed, one disc mislabeled bone.\\n        cm=confusion_matrix(np.array([0,0,0,0,0,1,2]),np.array([0,0,0,0,1,2,2]))\\n        np.testing.assert_array_equal(cm,[[4,0,0],[1,0,0],[0,1,1]])\\n        m=metrics(cm)\\n        self.assertAlmostEqual(m[\'pixel_accuracy\'],5/7)\\n        self.assertAlmostEqual(m[\'dice_macro_foreground\'],1/3)\\n        self.assertAlmostEqual(m[\'miou_foreground\'],1/4)\\n        self.assertAlmostEqual(m[\'legacy_mean_recall_background_vertebra\'],.5)\\n\\n    def test_ignore_and_absent_class(self):\\n        cm=confusion_matrix(np.array([0,2,1]),np.array([0,-100,1]))\\n        m=metrics(cm)\\n        self.assertEqual(m[\'valid_pixels\'],2)\\n        self.assertEqual(m[\'dice_macro_all\'],1)\\n        self.assertIsNone(m[\'dice_per_class\'][2])\\n\\n    def test_invalid_predictions_rejected(self):\\n        with self.assertRaises(ValueError):confusion_matrix(np.array([4]),np.array([1]))\\n\\n\\nif __name__==\'__main__\':unittest.main()\\n", "train.py": "\\"\\"\\"Fresh, traceable experiments on the supplied PNGs. See README.md for limits.\\"\\"\\"\\nimport argparse\\nimport contextlib\\nimport hashlib\\nimport io\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport random\\nimport subprocess\\nimport sys\\nimport time\\nimport numpy as np\\nfrom PIL import Image\\nimport torch\\nfrom torch import nn\\nfrom torch.utils.data import Dataset, DataLoader\\nfrom metrics import decode_mask, confusion_matrix, metrics\\n\\n\\nclass PngDataset(Dataset):\\n    def __init__(self, root, split, policy, limit=0):\\n        self.root, self.split, self.policy = root, split, policy\\n        self.files = sorted((root/split).glob(\'*.png\'), key=lambda p: int(p.stem))\\n        labels = {p.name for p in (root/(split+\'_labels\')).glob(\'*.png\')}\\n        if not self.files or {p.name for p in self.files} != labels:\\n            raise ValueError(\'Missing data or mismatched image/label filenames\')\\n        if limit:\\n            self.files = [self.files[i] for i in np.linspace(0, len(self.files)-1, min(limit,len(self.files)), dtype=int)]\\n    def __len__(self):\\n        return len(self.files)\\n    def __getitem__(self, i):\\n        p = self.files[i]\\n        with Image.open(p) as f:\\n            arr = np.array(f.convert(\'RGB\'), dtype=np.float32)/255\\n        with Image.open(self.root/(self.split+\'_labels\')/p.name) as f:\\n            mask = decode_mask(f, self.policy)\\n        if arr.shape != (512,512,3) or mask.shape != (512,512):\\n            raise ValueError(\'Native 512x512 inputs required; masks are never resized\')\\n        if not np.any(mask >= 0):\\n            raise ValueError(\'Mask has no valid labels\')\\n        arr = (arr-np.array([.485,.456,.406],dtype=np.float32))/np.array([.229,.224,.225],dtype=np.float32)\\n        return torch.from_numpy(arr.transpose(2,0,1).copy()), torch.from_numpy(mask), p.name\\n\\n\\ndef make_model(repo, name):\\n    base = next(repo.glob(\'*/J-Unet\'))\\n    sys.path.insert(0,str(base))\\n    if name == \'junet\':\\n        from nets.unet3plus import UNet_3Plus\\n        return UNet_3Plus(3)\\n    sys.path.insert(0,str(base.parent/\'UNET\'))\\n    from unet.unet_model import UNet\\n    return UNet(3,3)\\n\\n\\ndef dice_loss(logits, target):\\n    valid = target != -100\\n    onehot = nn.functional.one_hot(target.clamp_min(0),3).permute(0,3,1,2).float()\\n    prob = logits.float().softmax(1)\\n    mask = valid.unsqueeze(1)\\n    prob, onehot = prob*mask, onehot*mask\\n    score = (2*(prob*onehot).sum((2,3))+1)/((prob+onehot).sum((2,3))+1)\\n    return 1-score.mean()\\n\\n\\ndef atomic_save(obj,path):\\n    tmp = path.with_suffix(\'.tmp\')\\n    torch.save(obj,tmp); os.replace(tmp,path)\\n\\n\\ndef evaluate(model, loader, device, amp, output=None):\\n    model.eval()\\n    total = np.zeros((3,3),dtype=np.int64)\\n    rows = []\\n    with torch.inference_mode():\\n        for x,y,names in loader:\\n            with torch.autocast(device_type=device.type,enabled=amp):\\n                # Original baseline emits tensor sizes from forward().\\n                with contextlib.redirect_stdout(io.StringIO()):\\n                    pred = model(x.to(device)).argmax(1).cpu().numpy()\\n            for j,name in enumerate(names):\\n                cm = confusion_matrix(pred[j],y[j].numpy())\\n                total += cm\\n                rows.append(dict(filename=name,**metrics(cm)))\\n                if output:\\n                    Image.fromarray(np.array([0,100,255],dtype=np.uint8)[pred[j]]).save(output/name)\\n    result = metrics(total)\\n    result[\'images\'] = len(rows)\\n    result[\'image_macro_averages\'] = {k: float(np.mean([r[k] for r in rows if r[k] is not None]))\\n        for k in (\'dice_macro_all\',\'dice_macro_foreground\',\'miou_all\',\'miou_foreground\',\'pixel_accuracy\')}\\n    return result, rows\\n\\n\\ndef run(a):\\n    repo = Path(a.repo).resolve(); output = Path(a.output).resolve()\\n    output.mkdir(parents=True,exist_ok=True)\\n    if (output/\'config.json\').exists() and not a.resume:\\n        raise ValueError(\'Output already contains an experiment: choose a new directory or --resume\')\\n    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)\\n    torch.set_num_threads(2)\\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\\n    if device.type != \'cuda\' and not a.allow_cpu:\\n        raise RuntimeError(\'A Colab GPU is required. CPU training needs explicit --allow-cpu.\')\\n    if device.type == \'cuda\':\\n        torch.cuda.manual_seed_all(a.seed)\\n    torch.backends.cudnn.benchmark = False\\n    torch.backends.cudnn.deterministic = True\\n    root = next(repo.glob(\'*/data\'))\\n    train = PngDataset(root,\'train\',a.label_policy,a.train_limit)\\n    val = PngDataset(root,\'val\',a.label_policy,a.val_limit)\\n    test = PngDataset(root,\'test\',a.label_policy,a.test_limit)\\n    gen = torch.Generator().manual_seed(a.seed)\\n    tr_loader = DataLoader(train,batch_size=a.batch_size,shuffle=True,num_workers=0,generator=gen)\\n    va_loader = DataLoader(val,batch_size=1,shuffle=False,num_workers=0)\\n    te_loader = DataLoader(test,batch_size=1,shuffle=False,num_workers=0)\\n    model = make_model(repo,a.model).to(device)\\n    optimizer = torch.optim.Adam(model.parameters(),lr=a.lr)\\n    amp = device.type == \'cuda\' and not a.no_amp\\n    scaler = torch.amp.GradScaler(\'cuda\',enabled=amp)\\n    config = dict(vars(a),python=sys.version,torch=torch.__version__,numpy=np.__version__,\\n                  device=str(device),gpu=torch.cuda.get_device_name(0) if amp else None,\\n                  parameters=sum(p.numel() for p in model.parameters()),\\n                  counts={\'train\':len(train),\'val\':len(val),\'test\':len(test)},\\n                  aggregation=\'global confusion + image macro; all classes and foreground separately\',\\n                  limitations=[\'patient IDs unavailable\',\'PNG source not independently verified\',\\n                               \'not exact paper reproduction\',\'AMP changes numerical precision\'],\\n                  git_head=subprocess.check_output([\'git\',\'-C\',str(repo),\'rev-parse\',\'HEAD\'],text=True).strip(),\\n                  source_hashes={str(p.relative_to(repo)):hashlib.sha256(p.read_bytes()).hexdigest()\\n                      for p in list((repo/\'reproduce\').glob(\'*.py\'))+list(next(repo.glob(\'*/J-Unet/nets\')).glob(\'*.py\'))})\\n    start, best = 0, -1.0\\n    if a.resume:\\n        ckpt = torch.load(output/\'last.pt\',map_location=device,weights_only=False)\\n        prior = json.loads((output/\'config.json\').read_text())\\n        for key in (\'model\',\'seed\',\'label_policy\',\'batch_size\',\'lr\',\'train_limit\',\'val_limit\',\'test_limit\',\'no_amp\',\'loss\'):\\n            if prior[key] != config[key]: raise ValueError(f\'Resume configuration changed: {key}\')\\n        if prior[\'source_hashes\'] != config[\'source_hashes\']:\\n            raise ValueError(\'Source changed since checkpoint; start a new experiment\')\\n        model.load_state_dict(ckpt[\'model\']); optimizer.load_state_dict(ckpt[\'optimizer\'])\\n        scaler.load_state_dict(ckpt[\'scaler\']); gen.set_state(ckpt[\'loader_rng\'].cpu())\\n        torch.set_rng_state(ckpt[\'torch_rng\'].cpu())\\n        if device.type==\'cuda\': torch.cuda.set_rng_state_all([v.cpu() for v in ckpt[\'cuda_rng\']])\\n        start,best = ckpt[\'epoch\'],ckpt[\'best\']\\n    (output/\'config.json\').write_text(json.dumps(config,indent=2),encoding=\'utf-8\')\\n    print(json.dumps(config),flush=True)\\n    started=time.monotonic()\\n    for epoch in range(start,a.epochs):\\n        model.train()  # Restore BatchNorm training mode after every validation.\\n        losses=[]\\n        for step,(x,y,_) in enumerate(tr_loader):\\n            x,y=x.to(device),y.to(device)\\n            optimizer.zero_grad(set_to_none=True)\\n            with torch.autocast(device_type=device.type,enabled=amp):\\n                with contextlib.redirect_stdout(io.StringIO()): logits=model(x)\\n                loss = dice_loss(logits,y) if a.loss==\'dice\' else nn.functional.cross_entropy(logits,y,ignore_index=-100)\\n            if not torch.isfinite(loss): raise FloatingPointError(\'Nonfinite loss; no fabricated result or silent restart\')\\n            scaler.scale(loss).backward()\\n            scaler.step(optimizer); scaler.update()\\n            losses.append(loss.item())\\n            if step%25==0: print(f\'epoch={epoch+1} step={step+1}/{len(tr_loader)} loss={loss.item():.6f}\',flush=True)\\n        result,_ = evaluate(model,va_loader,device,amp)\\n        score=result[\'dice_macro_foreground\']\\n        improved = score is not None and score>best\\n        if improved:\\n            best=score\\n            atomic_save(dict(model=model.state_dict(),epoch=epoch+1),output/\'best.pt\')\\n        atomic_save(dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),\\n                         epoch=epoch+1,best=best,loader_rng=gen.get_state(),torch_rng=torch.get_rng_state(),\\n                         cuda_rng=torch.cuda.get_rng_state_all() if device.type==\'cuda\' else []),output/\'last.pt\')\\n        row=dict(epoch=epoch+1,train_loss=float(np.mean(losses)),validation=result,elapsed_seconds=time.monotonic()-started)\\n        with (output/\'history.jsonl\').open(\'a\',encoding=\'utf-8\') as f:f.write(json.dumps(row)+\'\\\\n\')\\n        print(json.dumps(row),flush=True)\\n    best_ckpt=torch.load(output/\'best.pt\',map_location=device,weights_only=True)\\n    model.load_state_dict(best_ckpt[\'model\'])\\n    pred_dir=output/\'predictions\';pred_dir.mkdir(exist_ok=True)\\n    result,rows=evaluate(model,te_loader,device,amp,pred_dir)\\n    result.update(best_epoch=best_ckpt[\'epoch\'],label_policy=a.label_policy,\\n                  experiment=\'new_training_on_repository_PNGs\',\\n                  full_dataset=not any((a.train_limit,a.val_limit,a.test_limit)),\\n                  planned_epochs=a.epochs,source_checkpoint=\'best.pt\',\\n                  patient_independent_test=\'unverified\')\\n    (output/\'test_metrics.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\\n    (output/\'test_per_image.json\').write_text(json.dumps(rows,indent=2),encoding=\'utf-8\')\\n    print(\'TEST_RESULT \'+json.dumps(result),flush=True)\\n\\n\\nif __name__==\'__main__\':\\n    p=argparse.ArgumentParser()\\n    p.add_argument(\'--repo\',default=str(Path(__file__).resolve().parents[1]))\\n    p.add_argument(\'--output\',required=True)\\n    p.add_argument(\'--model\',choices=[\'junet\',\'unet\'],default=\'junet\')\\n    p.add_argument(\'--epochs\',type=int,default=100);p.add_argument(\'--batch-size\',type=int,default=2)\\n    p.add_argument(\'--lr\',type=float,default=1e-4);p.add_argument(\'--seed\',type=int,default=20260915)\\n    p.add_argument(\'--loss\',choices=[\'dice\',\'ce\'],default=\'dice\')\\n    p.add_argument(\'--label-policy\',choices=[\'ignore\',\'legacy-background\'],default=\'ignore\')\\n    for split in (\'train\',\'val\',\'test\'):p.add_argument(f\'--{split}-limit\',type=int,default=0)\\n    p.add_argument(\'--resume\',action=\'store_true\');p.add_argument(\'--no-amp\',action=\'store_true\')\\n    p.add_argument(\'--allow-cpu\',action=\'store_true\')\\n    a=p.parse_args()\\n    if a.epochs<1 or a.batch_size<1 or min(a.train_limit,a.val_limit,a.test_limit)<0:p.error(\'Invalid count\')\\n    run(a)\\n"}')
code_dir=REPO/'reproduce'; code_dir.mkdir(exist_ok=True)
for name,source in payload.items():
    (code_dir/name).write_text(source,encoding='utf-8')
subprocess.run([sys.executable,str(code_dir/'test_core.py')],check=True)
print('指标与标签测试通过')


## 输出位置与训练配置
默认保存在 Colab `/content`；运行时释放后会丢失，请下载结果。长期训练可将 USE_DRIVE 改为 True 并自行完成 Drive 授权。续跑需相同代码、配置和 RUN_NAME，并设置 RESUME=True。

In [ ]:
USE_DRIVE=False
MODEL='junet'  # 也支持原仓库 unet 基线
EPOCHS=100
LABEL_POLICY='ignore'  # 或 legacy-background；两者都无法恢复原始标注
RESUME=False
RUN_NAME='junet_100_ignore_seed20260915'  # 基线/敏感性分析请使用不同名字
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT=Path('/content/drive/MyDrive/Spine-Re-experiments')
else:
    OUTPUT_ROOT=Path('/content/spine_runs')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
RUN_DIR=OUTPUT_ROOT/RUN_NAME
print('结果保存到',RUN_DIR)


## 审计数据
检查图像/标签配对、尺寸、完全重复、未知标签像素，并重算旧预测图。此步骤没有训练模型。

In [ ]:
audit_result=subprocess.run([sys.executable,str(code_dir/'audit.py'),'--repo',str(REPO),'--output',str(OUTPUT_ROOT/'audit')],capture_output=True,text=True)
print(audit_result.stdout)
if audit_result.returncode:
    print(audit_result.stderr)
    raise RuntimeError('数据审计失败')
audit=json.loads((OUTPUT_ROOT/'audit/data_audit.json').read_text())
assert not audit['cross_split_duplicate_groups'], '发现跨集合完全重复，先解决划分问题'


## 训练并在结束后测试
每轮保存 last.pt（可续跑），按验证集前景 Dice 保存 best.pt。测试集不参与模型选择。T4 上完整训练可能需长时间或跨多个会话续跑。

In [ ]:
cmd=[sys.executable,'-u',str(code_dir/'train.py'),'--repo',str(REPO),'--output',str(RUN_DIR),'--model',MODEL,'--epochs',str(EPOCHS),'--label-policy',LABEL_POLICY]
if RESUME:cmd.append('--resume')
RUN_DIR.mkdir(parents=True,exist_ok=True)
with (RUN_DIR/'console.log').open('a',encoding='utf-8') as log:
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end='');log.write(line);log.flush()
    returncode=proc.wait()
if returncode:raise RuntimeError(f'训练失败，退出码 {returncode}；请查看 console.log')
result=json.loads((RUN_DIR/'test_metrics.json').read_text())
print(json.dumps(result,indent=2,ensure_ascii=False))


## 保存结果
ZIP 包含日志、配置、逐图指标、预测图和模型权重，可较大。机器断开前下载，或者保存在已挂载 Drive。

In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/'+RUN_NAME,'zip',RUN_DIR)
files.download(archive)
